# Global carbon-benefit raster across existing forest areas

This notebook applies the same Jamaica-clipped global carbon-benefit raster used for the mangrove and forest-restoration notebooks to the existing forest footprint from DPhil paper 2.

Method:
1. Read the DPhil paper 2 land-use layer.
2. Select natural existing forest classes plus the existing-forest share of mixed classes using `Robyn_catchment_analysis.existing_forest_classes` and `Robyn_catchment_analysis.mixed_land_use_fractions`.
3. Calculate patch-level zonal statistics from the carbon-benefit raster.
4. Fill patches with no valid raster cells from the nearest observed existing-forest patch.
5. Save patch, land-use, parish and catchment summaries for the mapping notebook.

Important interpretation note: the raster is treated consistently with the mangrove/restoration notebooks as source carbon-benefit values per hectare. Applying it to existing forest is an indicative overlay of the same raster across the existing-forest footprint; it should not be reported as measured current forest carbon stock unless the source raster documentation supports that interpretation.

In [ ]:
import os
import sys
from pathlib import Path

base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
matplotlib_config_dir = base_path / ".matplotlib"
matplotlib_config_dir.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(matplotlib_config_dir))

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from rasterio.errors import WindowError
from rasterio.features import geometry_mask, geometry_window
from shapely.geometry import LineString

robyn_libraries_path = (base_path / "robyns_libraries").resolve()
if str(robyn_libraries_path) not in sys.path:
    sys.path.append(str(robyn_libraries_path))
import Robyn_catchment_analysis
import Robyn_paper_2_defs

pd.set_option("display.max_columns", 200)

## Inputs and outputs

In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"
carbon_benefit_units_assumed = "tonnes C per hectare"
include_all_touched_pixels = False

source_carbon_output_dir = base_path / "dphil_paper_3/processed_data/carbon/global_carbon_benefit"
output_dir = base_path / "dphil_paper_3/processed_data/carbon/global_carbon_benefit_existing_forests"
figure_dir = base_path / "dphil_paper_3/results/co_benefits/carbon/global_carbon_benefit_existing_forests"
output_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

global_carbon_projected_path = source_carbon_output_dir / "global_carbon_benefit_jamaica_20km_buffer_epsg3448.tif"
jamaica_boundary_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg"
admin_boundaries_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/admin_boundaries.gpkg"
existing_forest_land_use_path = base_path / "dphil_paper_2/processed_data/land_use_forest_and_afforestable.gpkg"
catchments_path = base_path / "dphil_paper_2/processed_data/major_river_catchments/major_basins_plus_coastal_unionized_final.gpkg"

patch_summary_csv_path = output_dir / "existing_forest_patch_global_carbon_benefit_summary.csv"
patch_summary_gpkg_path = output_dir / "existing_forest_patch_global_carbon_benefit_summary.gpkg"
nearest_neighbor_summary_csv_path = output_dir / "existing_forest_patch_global_carbon_benefit_nearest_neighbor_fill.csv"
nearest_neighbor_summary_gpkg_path = output_dir / "existing_forest_patch_global_carbon_benefit_nearest_neighbor_fill.gpkg"
nearest_neighbor_links_gpkg_path = output_dir / "existing_forest_patch_global_carbon_benefit_nearest_neighbor_links.gpkg"
patch_table_path = output_dir / "existing_forest_patch_global_carbon_benefit_table.csv"
method_summary_csv_path = output_dir / "existing_forest_global_carbon_benefit_method_summary.csv"
land_use_summary_csv_path = output_dir / "existing_forest_global_carbon_benefit_by_land_use_class.csv"
parish_summary_csv_path = output_dir / "existing_forest_global_carbon_benefit_by_parish.csv"
catchment_summary_csv_path = output_dir / "existing_forest_global_carbon_benefit_by_catchment.csv"
catchment_summary_gpkg_path = output_dir / "existing_forest_global_carbon_benefit_by_catchment.gpkg"

patch_map_path = figure_dir / "existing_forest_patch_global_carbon_benefit_map.png"
no_coverage_map_path = figure_dir / "existing_forest_patch_global_carbon_benefit_no_valid_cells_map.png"
nearest_neighbor_source_map_path = figure_dir / "existing_forest_patch_global_carbon_benefit_nearest_neighbor_sources_map.png"
land_use_bar_chart_path = figure_dir / "existing_forest_global_carbon_benefit_by_land_use_class.png"
catchment_map_path = figure_dir / "existing_forest_global_carbon_benefit_by_catchment_map.png"

required_paths = [
    global_carbon_projected_path,
    jamaica_boundary_path,
    admin_boundaries_path,
    existing_forest_land_use_path,
    catchments_path,
]
for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing required input: {required_path}")

print(f"Existing forest land-use layer: {existing_forest_land_use_path}")
print(f"Projected Jamaica carbon raster: {global_carbon_projected_path}")
print(f"Outputs: {output_dir}")

## Load the existing forest footprint

In [ ]:
def get_existing_forest_fraction(land_use_class):
    if land_use_class in Robyn_catchment_analysis.existing_forest_classes:
        return 1.0
    if land_use_class in Robyn_catchment_analysis.mixed_land_use_fractions:
        mixed_class_fractions = Robyn_catchment_analysis.mixed_land_use_fractions[land_use_class]
        return float(mixed_class_fractions.get("existing_forest_classes", 0.0))
    return 0.0


def describe_existing_forest_source(land_use_class):
    if land_use_class in Robyn_catchment_analysis.existing_forest_classes:
        return "Natural existing forest class"
    if get_existing_forest_fraction(land_use_class) > 0:
        return "Existing-forest share of mixed class"
    return "Not included"


jamaica_boundary = gpd.read_file(jamaica_boundary_path).to_crs(jamaica_metric_grid_crs)
jamaica_boundary = jamaica_boundary[jamaica_boundary.geometry.notna() & ~jamaica_boundary.geometry.is_empty].copy()

land_use = gpd.read_file(existing_forest_land_use_path).to_crs(jamaica_metric_grid_crs)
land_use = land_use[land_use.geometry.notna() & ~land_use.geometry.is_empty].copy()

required_land_use_columns = ["OBJECTID", "Classify", "LU_CODE", "forest_flood_equivalent_values", "geometry"]
missing_land_use_columns = [column for column in required_land_use_columns if column not in land_use.columns]
if missing_land_use_columns:
    raise ValueError(f"Missing required columns in land-use layer: {missing_land_use_columns}")

land_use["existing_forest_fraction"] = land_use["Classify"].apply(get_existing_forest_fraction)
land_use["existing_forest_source"] = land_use["Classify"].apply(describe_existing_forest_source)
existing_forest_patches = land_use[land_use["existing_forest_fraction"] > 0].copy().reset_index(drop=True)
if existing_forest_patches.empty:
    raise ValueError("No existing forest polygons found from the paper-2 existing forest class definitions.")

if existing_forest_patches["OBJECTID"].notna().all() and existing_forest_patches["OBJECTID"].is_unique:
    existing_forest_patches["Existing_forest_patch_id"] = existing_forest_patches["OBJECTID"].astype(int)
else:
    existing_forest_patches["Existing_forest_patch_id"] = np.arange(1, len(existing_forest_patches) + 1)

existing_forest_patches["geometry_is_valid"] = existing_forest_patches.geometry.is_valid
existing_forest_patches["patch_area_m2"] = existing_forest_patches.geometry.area
existing_forest_patches["patch_area_ha"] = existing_forest_patches["patch_area_m2"] / 10_000
existing_forest_patches["existing_forest_area_ha"] = (
    existing_forest_patches["patch_area_ha"] * existing_forest_patches["existing_forest_fraction"]
)

parishes = gpd.read_file(admin_boundaries_path, layer="admin1").to_crs(jamaica_metric_grid_crs)
parishes = parishes[["PARISH", "geometry"]].copy()
parishes["geometry"] = parishes.geometry.make_valid()

existing_forest_points = existing_forest_patches.copy()
existing_forest_points["geometry"] = existing_forest_points.geometry.representative_point()
patch_parishes = gpd.sjoin(
    existing_forest_points[["Existing_forest_patch_id", "geometry"]],
    parishes,
    how="left",
    predicate="within",
).drop(columns=["index_right"])
existing_forest_patches = existing_forest_patches.merge(
    patch_parishes[["Existing_forest_patch_id", "PARISH"]],
    on="Existing_forest_patch_id",
    how="left",
).rename(columns={"PARISH": "representative_parish"})

broad_flood_equivalent_area_ha = (
    land_use["forest_flood_equivalent_values"].astype(float) * land_use.geometry.area / 10_000
).sum()
footprint_summary = pd.DataFrame(
    [
        {"metric": "all_land_use_polygons", "value": len(land_use)},
        {"metric": "existing_forest_patches", "value": len(existing_forest_patches)},
        {"metric": "invalid_existing_forest_patch_geometries", "value": int((~existing_forest_patches["geometry_is_valid"]).sum())},
        {"metric": "unweighted_existing_forest_patch_area_ha", "value": existing_forest_patches["patch_area_ha"].sum()},
        {"metric": "weighted_existing_forest_area_ha", "value": existing_forest_patches["existing_forest_area_ha"].sum()},
        {"metric": "fully_existing_forest_area_ha", "value": existing_forest_patches.loc[existing_forest_patches["existing_forest_fraction"] == 1.0, "existing_forest_area_ha"].sum()},
        {"metric": "mixed_existing_forest_area_ha", "value": existing_forest_patches.loc[existing_forest_patches["existing_forest_fraction"] < 1.0, "existing_forest_area_ha"].sum()},
        {"metric": "broad_flood_equivalent_area_ha_not_used_for_this_notebook", "value": broad_flood_equivalent_area_ha},
    ]
)

class_definition_summary = (
    existing_forest_patches.groupby(["Classify", "LU_CODE", "existing_forest_source", "existing_forest_fraction"], dropna=False)
    .agg(
        patch_count=("Existing_forest_patch_id", "count"),
        patch_area_ha=("patch_area_ha", "sum"),
        existing_forest_area_ha=("existing_forest_area_ha", "sum"),
    )
    .reset_index()
    .sort_values("existing_forest_area_ha", ascending=False)
)

display(footprint_summary)
display(class_definition_summary)
display(
    existing_forest_patches[
        [
            "Existing_forest_patch_id",
            "Classify",
            "LU_CODE",
            "representative_parish",
            "existing_forest_fraction",
            "patch_area_ha",
            "existing_forest_area_ha",
            "geometry_is_valid",
        ]
    ].head()
)

## Inspect the carbon raster

In [ ]:
with rasterio.open(global_carbon_projected_path) as carbon_projected:
    carbon_nodata = carbon_projected.nodata
    pixel_area_ha = abs(carbon_projected.transform.a * carbon_projected.transform.e) / 10_000
    carbon_metadata = pd.DataFrame(
        [
            {"property": "carbon_raster_path", "value": str(global_carbon_projected_path)},
            {"property": "carbon_crs", "value": str(carbon_projected.crs)},
            {"property": "carbon_shape", "value": f"{carbon_projected.height:,} rows × {carbon_projected.width:,} columns"},
            {"property": "carbon_resolution_m", "value": abs(carbon_projected.transform.a)},
            {"property": "carbon_pixel_area_ha", "value": pixel_area_ha},
            {"property": "carbon_nodata", "value": carbon_nodata},
            {"property": "units_assumed", "value": carbon_benefit_units_assumed},
            {"property": "interpretation_note", "value": "Indicative overlay; not measured current forest carbon stock."},
        ]
    )

display(carbon_metadata)

## Calculate patch-level carbon potential

In [ ]:
def append_empty_zonal_statistics(patch_rows, id_column, patch_id, error_text=""):
    patch_rows.append(
        {
            id_column: patch_id,
            "valid_carbon_benefit_pixel_count": 0,
            "min_carbon_benefit_value": np.nan,
            "mean_carbon_benefit_value": np.nan,
            "max_carbon_benefit_value": np.nan,
            "sum_carbon_benefit_values": np.nan,
            "zonal_stat_error": error_text,
        }
    )


def calculate_patch_zonal_statistics(patches, id_column, raster_path, nodata_value, all_touched):
    patch_rows = []

    with rasterio.open(raster_path) as carbon_raster:
        for patch_id, patch_geometry in patches[[id_column, "geometry"]].itertuples(index=False, name=None):
            try:
                patch_window = geometry_window(carbon_raster, [patch_geometry])
                raster_values = carbon_raster.read(1, window=patch_window, masked=False)
                patch_mask = geometry_mask(
                    [patch_geometry],
                    out_shape=raster_values.shape,
                    transform=carbon_raster.window_transform(patch_window),
                    invert=True,
                    all_touched=all_touched,
                )
                valid_pixel_mask = patch_mask & np.isfinite(raster_values)
                if nodata_value is not None:
                    valid_pixel_mask = valid_pixel_mask & (raster_values != nodata_value)

                patch_values = raster_values[valid_pixel_mask]
                if patch_values.size == 0:
                    append_empty_zonal_statistics(patch_rows, id_column, patch_id)
                else:
                    patch_rows.append(
                        {
                            id_column: patch_id,
                            "valid_carbon_benefit_pixel_count": int(patch_values.size),
                            "min_carbon_benefit_value": float(patch_values.min()),
                            "mean_carbon_benefit_value": float(patch_values.mean()),
                            "max_carbon_benefit_value": float(patch_values.max()),
                            "sum_carbon_benefit_values": float(patch_values.sum()),
                            "zonal_stat_error": "",
                        }
                    )
            except WindowError:
                append_empty_zonal_statistics(patch_rows, id_column, patch_id)
            except Exception as zonal_error:
                append_empty_zonal_statistics(patch_rows, id_column, patch_id, str(zonal_error)[:200])

    return pd.DataFrame(patch_rows)


carbon_statistics = calculate_patch_zonal_statistics(
    existing_forest_patches,
    "Existing_forest_patch_id",
    global_carbon_projected_path,
    carbon_nodata,
    include_all_touched_pixels,
)

patch_columns = [
    "Existing_forest_patch_id",
    "OBJECTID",
    "Classify",
    "LU_CODE",
    "existing_forest_source",
    "representative_parish",
    "existing_forest_fraction",
    "patch_area_m2",
    "patch_area_ha",
    "existing_forest_area_ha",
    "geometry_is_valid",
    "geometry",
]
carbon_benefit_summary = existing_forest_patches[patch_columns].merge(
    carbon_statistics,
    on="Existing_forest_patch_id",
    how="left",
)
carbon_benefit_summary["valid_carbon_benefit_pixel_count"] = carbon_benefit_summary[
    "valid_carbon_benefit_pixel_count"
].fillna(0).astype(int)
carbon_benefit_summary["has_observed_carbon_benefit"] = carbon_benefit_summary[
    "valid_carbon_benefit_pixel_count"
] > 0
carbon_benefit_summary["carbon_benefit_status"] = np.where(
    carbon_benefit_summary["has_observed_carbon_benefit"],
    "Observed raster overlap",
    "No valid raster cells",
)
carbon_benefit_summary["coverage_area_ha"] = carbon_benefit_summary["valid_carbon_benefit_pixel_count"] * pixel_area_ha
carbon_benefit_summary["coverage_pct_of_patch"] = np.where(
    carbon_benefit_summary["patch_area_ha"] > 0,
    carbon_benefit_summary["coverage_area_ha"] / carbon_benefit_summary["patch_area_ha"] * 100,
    np.nan,
)
# The raster value is interpreted as tonnes C/ha. Pixel area and summed pixel values are retained only
# as diagnostics; carbon totals are density multiplied by the actual existing-forest area of each patch.
carbon_benefit_summary["carbon_potential_per_existing_forest_ha"] = carbon_benefit_summary[
    "mean_carbon_benefit_value"
].where(carbon_benefit_summary["has_observed_carbon_benefit"])
carbon_benefit_summary["unweighted_total_carbon_benefit_source_units"] = (
    carbon_benefit_summary["carbon_potential_per_existing_forest_ha"] * carbon_benefit_summary["patch_area_ha"]
)
carbon_benefit_summary["total_carbon_potential_source_units"] = (
    carbon_benefit_summary["carbon_potential_per_existing_forest_ha"]
    * carbon_benefit_summary["existing_forest_area_ha"]
)

observed_display_columns = [
    "Existing_forest_patch_id",
    "Classify",
    "LU_CODE",
    "representative_parish",
    "existing_forest_fraction",
    "existing_forest_area_ha",
    "valid_carbon_benefit_pixel_count",
    "coverage_pct_of_patch",
    "mean_carbon_benefit_value",
    "total_carbon_potential_source_units",
    "carbon_potential_per_existing_forest_ha",
    "zonal_stat_error",
]
display(
    carbon_benefit_summary.sort_values("total_carbon_potential_source_units", ascending=False)[observed_display_columns].head(20)
)

## Fill patches with no raster coverage from nearest observed patches

In [ ]:
observed_carbon_patches = carbon_benefit_summary[carbon_benefit_summary["has_observed_carbon_benefit"]].copy()
missing_carbon_patches = carbon_benefit_summary[~carbon_benefit_summary["has_observed_carbon_benefit"]].copy()

if observed_carbon_patches.empty:
    raise ValueError("No existing forest patches have observed global carbon-benefit raster coverage.")

nearest_neighbor_fill_columns = [
    "Existing_forest_patch_id",
    "nearest_source_existing_forest_patch_id",
    "nearest_source_classify",
    "nearest_source_lu_code",
    "nearest_source_parish",
    "nearest_source_distance_m",
    "nearest_source_carbon_potential_per_existing_forest_ha",
    "nn_total_carbon_potential_source_units",
]

if missing_carbon_patches.empty:
    nearest_neighbor_fill = pd.DataFrame(columns=nearest_neighbor_fill_columns)
else:
    missing_patch_targets = missing_carbon_patches[["Existing_forest_patch_id", "existing_forest_area_ha", "geometry"]].copy()
    observed_patch_sources = observed_carbon_patches[
        [
            "Existing_forest_patch_id",
            "Classify",
            "LU_CODE",
            "representative_parish",
            "carbon_potential_per_existing_forest_ha",
            "geometry",
        ]
    ].rename(
        columns={
            "Existing_forest_patch_id": "nearest_source_existing_forest_patch_id",
            "Classify": "nearest_source_classify",
            "LU_CODE": "nearest_source_lu_code",
            "representative_parish": "nearest_source_parish",
            "carbon_potential_per_existing_forest_ha": "nearest_source_carbon_potential_per_existing_forest_ha",
        }
    )
    nearest_neighbor_join = gpd.sjoin_nearest(
        missing_patch_targets,
        observed_patch_sources,
        how="left",
        distance_col="nearest_source_distance_m",
    ).drop(columns=["index_right"])
    nearest_neighbor_join = (
        nearest_neighbor_join.sort_values(
            [
                "Existing_forest_patch_id",
                "nearest_source_distance_m",
                "nearest_source_existing_forest_patch_id",
            ]
        )
        .drop_duplicates("Existing_forest_patch_id", keep="first")
        .copy()
    )
    nearest_neighbor_join["nn_total_carbon_potential_source_units"] = (
        nearest_neighbor_join["nearest_source_carbon_potential_per_existing_forest_ha"]
        * nearest_neighbor_join["existing_forest_area_ha"]
    )
    nearest_neighbor_fill = pd.DataFrame(nearest_neighbor_join[nearest_neighbor_fill_columns])

carbon_benefit_summary = carbon_benefit_summary.merge(
    nearest_neighbor_fill,
    on="Existing_forest_patch_id",
    how="left",
)
carbon_benefit_summary["nearest_source_distance_km"] = carbon_benefit_summary["nearest_source_distance_m"] / 1_000
carbon_benefit_summary["carbon_potential_estimate_method"] = np.where(
    carbon_benefit_summary["has_observed_carbon_benefit"],
    "observed_raster_overlap",
    "nearest_observed_patch_mean",
)
carbon_benefit_summary["carbon_potential_per_existing_forest_ha_with_nn_fill"] = carbon_benefit_summary[
    "carbon_potential_per_existing_forest_ha"
].where(
    carbon_benefit_summary["has_observed_carbon_benefit"],
    carbon_benefit_summary["nearest_source_carbon_potential_per_existing_forest_ha"],
)
carbon_benefit_summary["total_carbon_potential_with_nn_fill"] = carbon_benefit_summary[
    "total_carbon_potential_source_units"
].where(
    carbon_benefit_summary["has_observed_carbon_benefit"],
    carbon_benefit_summary["nn_total_carbon_potential_source_units"],
)

nearest_neighbor_fill_gdf = carbon_benefit_summary[~carbon_benefit_summary["has_observed_carbon_benefit"]].copy()
nearest_neighbor_fill_output_columns = [
    "Existing_forest_patch_id",
    "Classify",
    "LU_CODE",
    "representative_parish",
    "existing_forest_area_ha",
    "nearest_source_existing_forest_patch_id",
    "nearest_source_classify",
    "nearest_source_lu_code",
    "nearest_source_parish",
    "nearest_source_distance_m",
    "nearest_source_distance_km",
    "nearest_source_carbon_potential_per_existing_forest_ha",
    "nn_total_carbon_potential_source_units",
]

nearest_neighbor_link_rows = []
if len(nearest_neighbor_fill_gdf) > 0:
    nearest_neighbor_source_lookup = observed_carbon_patches.set_index("Existing_forest_patch_id")
    for filled_patch in nearest_neighbor_fill_gdf.itertuples(index=False):
        source_patch = nearest_neighbor_source_lookup.loc[int(filled_patch.nearest_source_existing_forest_patch_id)]
        nearest_neighbor_link_rows.append(
            {
                "Existing_forest_patch_id": int(filled_patch.Existing_forest_patch_id),
                "nearest_source_existing_forest_patch_id": int(filled_patch.nearest_source_existing_forest_patch_id),
                "nearest_source_distance_m": filled_patch.nearest_source_distance_m,
                "nearest_source_distance_km": filled_patch.nearest_source_distance_km,
                "geometry": LineString(
                    [
                        filled_patch.geometry.representative_point(),
                        source_patch.geometry.representative_point(),
                    ]
                ),
            }
        )
nearest_neighbor_links = gpd.GeoDataFrame(nearest_neighbor_link_rows, crs=carbon_benefit_summary.crs)

observed_total_carbon_potential = carbon_benefit_summary["total_carbon_potential_source_units"].fillna(0)
nearest_neighbor_total_carbon_potential = carbon_benefit_summary[
    "nn_total_carbon_potential_source_units"
].where(~carbon_benefit_summary["has_observed_carbon_benefit"])
patch_table = pd.DataFrame(
    {
        "existing_forest_patch_id": carbon_benefit_summary["Existing_forest_patch_id"].astype(int),
        "land_use_class": carbon_benefit_summary["Classify"],
        "land_use_code": carbon_benefit_summary["LU_CODE"],
        "representative_parish": carbon_benefit_summary["representative_parish"],
        "existing_forest_fraction": carbon_benefit_summary["existing_forest_fraction"],
        "patch_size_hectares": carbon_benefit_summary["patch_area_ha"],
        "existing_forest_area_hectares": carbon_benefit_summary["existing_forest_area_ha"],
        "global_carbon_potential_value": observed_total_carbon_potential,
        "global_carbon_potential_value_per_existing_forest_hectare": np.where(
            carbon_benefit_summary["existing_forest_area_ha"] > 0,
            observed_total_carbon_potential / carbon_benefit_summary["existing_forest_area_ha"],
            np.nan,
        ),
        "distance_to_nearest_neighbour_km_for_no_data": carbon_benefit_summary["nearest_source_distance_km"].where(
            ~carbon_benefit_summary["has_observed_carbon_benefit"]
        ),
        "carbon_potential_from_nearest_neighbour": nearest_neighbor_total_carbon_potential,
        "nearest_neighbour_source_carbon_potential_per_hectare_used": carbon_benefit_summary[
            "nearest_source_carbon_potential_per_existing_forest_ha"
        ].where(~carbon_benefit_summary["has_observed_carbon_benefit"]),
        "carbon_potential_from_nearest_neighbour_per_hectare": np.where(
            carbon_benefit_summary["existing_forest_area_ha"] > 0,
            nearest_neighbor_total_carbon_potential / carbon_benefit_summary["existing_forest_area_ha"],
            np.nan,
        ),
        "global_carbon_potential_value_with_nearest_neighbour_fill": carbon_benefit_summary[
            "total_carbon_potential_with_nn_fill"
        ],
        "global_carbon_potential_per_existing_forest_hectare_with_nearest_neighbour_fill": carbon_benefit_summary[
            "carbon_potential_per_existing_forest_ha_with_nn_fill"
        ],
    }
).sort_values("existing_forest_patch_id")
patch_table_decimal_columns = [column for column in patch_table.columns if column != "existing_forest_patch_id"]
for decimal_column in patch_table_decimal_columns:
    if pd.api.types.is_numeric_dtype(patch_table[decimal_column]):
        patch_table[decimal_column] = patch_table[decimal_column].round(2)

method_summary = pd.DataFrame(
    [
        {"metric": "existing_forest_patches", "value": int(len(carbon_benefit_summary))},
        {"metric": "patches_with_observed_raster_coverage", "value": int(carbon_benefit_summary["has_observed_carbon_benefit"].sum())},
        {"metric": "patches_filled_by_nearest_neighbor", "value": int(len(nearest_neighbor_fill_gdf))},
        {"metric": "weighted_existing_forest_area_ha", "value": float(carbon_benefit_summary["existing_forest_area_ha"].sum())},
        {"metric": "observed_total_carbon_potential_tonnes_c", "value": float(carbon_benefit_summary["total_carbon_potential_source_units"].sum(skipna=True))},
        {"metric": "nearest_neighbor_filled_total_carbon_potential_tonnes_c", "value": float(carbon_benefit_summary["total_carbon_potential_with_nn_fill"].sum(skipna=True))},
        {"metric": "mean_carbon_potential_per_existing_forest_ha_with_nn_fill", "value": float(carbon_benefit_summary["total_carbon_potential_with_nn_fill"].sum(skipna=True) / carbon_benefit_summary["existing_forest_area_ha"].sum())},
        {"metric": "carbon_density_method", "value": "Observed patches use mean raster tonnes C/ha; totals equal density multiplied by actual existing-forest hectares."},
        {"metric": "nearest_neighbor_method", "value": "Missing patches inherit nearest observed patch tonnes C/ha; totals equal inherited density multiplied by actual existing-forest hectares."},
        {"metric": "interpretation_note", "value": "Indicative overlay of carbon-benefit raster across existing forest, not measured stock."},
    ]
)

display(method_summary)
display(patch_table.head(20))

## Summaries by land-use class, parish and catchment

In [ ]:
def summarize_patch_values_by_column(patches, group_column):
    zone_summary = (
        patches.groupby(group_column, dropna=False)
        .agg(
            patch_count=("Existing_forest_patch_id", "count"),
            observed_patch_count=("has_observed_carbon_benefit", "sum"),
            existing_forest_area_ha=("existing_forest_area_ha", "sum"),
            total_carbon_potential_source_units=("total_carbon_potential_source_units", "sum"),
            total_carbon_potential_with_nn_fill=("total_carbon_potential_with_nn_fill", "sum"),
        )
        .reset_index()
    )
    zone_summary["filled_patch_count"] = zone_summary["patch_count"] - zone_summary["observed_patch_count"]
    zone_summary["carbon_potential_per_existing_forest_ha_with_nn_fill"] = np.where(
        zone_summary["existing_forest_area_ha"] > 0,
        zone_summary["total_carbon_potential_with_nn_fill"] / zone_summary["existing_forest_area_ha"],
        np.nan,
    )
    return zone_summary


land_use_summary = (
    carbon_benefit_summary.groupby(
        ["Classify", "LU_CODE", "existing_forest_source", "existing_forest_fraction"],
        dropna=False,
    )
    .agg(
        patch_count=("Existing_forest_patch_id", "count"),
        observed_patch_count=("has_observed_carbon_benefit", "sum"),
        patch_area_ha=("patch_area_ha", "sum"),
        existing_forest_area_ha=("existing_forest_area_ha", "sum"),
        total_carbon_potential_source_units=("total_carbon_potential_source_units", "sum"),
        total_carbon_potential_with_nn_fill=("total_carbon_potential_with_nn_fill", "sum"),
    )
    .reset_index()
)
land_use_summary["filled_patch_count"] = land_use_summary["patch_count"] - land_use_summary["observed_patch_count"]
land_use_summary["carbon_potential_per_existing_forest_ha_with_nn_fill"] = np.where(
    land_use_summary["existing_forest_area_ha"] > 0,
    land_use_summary["total_carbon_potential_with_nn_fill"] / land_use_summary["existing_forest_area_ha"],
    np.nan,
)
land_use_summary = land_use_summary.sort_values("total_carbon_potential_with_nn_fill", ascending=False)

parish_summary = summarize_patch_values_by_column(carbon_benefit_summary, "representative_parish")
parish_summary = parish_summary.rename(columns={"representative_parish": "PARISH"})
parish_summary = parish_summary.sort_values("total_carbon_potential_with_nn_fill", ascending=False)

catchments = gpd.read_file(catchments_path).to_crs(jamaica_metric_grid_crs)
catchments = catchments[catchments.geometry.notna() & ~catchments.geometry.is_empty].copy()
catchment_points = carbon_benefit_summary[["Existing_forest_patch_id", "geometry"]].copy()
catchment_points["geometry"] = catchment_points.geometry.representative_point()
patch_catchments = gpd.sjoin(
    catchment_points,
    catchments[["catchment_uid", "geometry"]],
    how="left",
    predicate="within",
).drop(columns=["index_right"])
carbon_benefit_summary = carbon_benefit_summary.merge(
    patch_catchments[["Existing_forest_patch_id", "catchment_uid"]],
    on="Existing_forest_patch_id",
    how="left",
)
catchment_summary = summarize_patch_values_by_column(carbon_benefit_summary, "catchment_uid")
catchment_summary = catchment_summary.sort_values("total_carbon_potential_with_nn_fill", ascending=False)

catchment_summary_gdf = catchments.merge(catchment_summary, on="catchment_uid", how="left")
summary_numeric_columns = catchment_summary_gdf.select_dtypes(include=["number"]).columns
catchment_summary_gdf[summary_numeric_columns] = catchment_summary_gdf[summary_numeric_columns].fillna(0)

zone_method_note = pd.DataFrame(
    [
        {
            "summary": "parish_and_catchment",
            "allocation_method": "Patch totals assigned by representative point, not split by exact polygon-zone overlay.",
        }
    ]
)
display(zone_method_note)
display(land_use_summary)
display(parish_summary)
display(catchment_summary.head(20))

## Save outputs

In [ ]:
patch_summary_columns = [column for column in carbon_benefit_summary.columns if column != "geometry"]
carbon_benefit_summary[patch_summary_columns].to_csv(patch_summary_csv_path, index=False)
carbon_benefit_summary.to_file(patch_summary_gpkg_path, driver="GPKG")
patch_table.to_csv(patch_table_path, index=False, float_format="%.2f")
nearest_neighbor_fill_gdf[nearest_neighbor_fill_output_columns].to_csv(nearest_neighbor_summary_csv_path, index=False)
if len(nearest_neighbor_fill_gdf) > 0:
    nearest_neighbor_fill_gdf.to_file(nearest_neighbor_summary_gpkg_path, driver="GPKG")
if len(nearest_neighbor_links) > 0:
    nearest_neighbor_links.to_file(nearest_neighbor_links_gpkg_path, driver="GPKG")
method_summary.to_csv(method_summary_csv_path, index=False)
land_use_summary.to_csv(land_use_summary_csv_path, index=False)
parish_summary.to_csv(parish_summary_csv_path, index=False)
catchment_summary.to_csv(catchment_summary_csv_path, index=False)
catchment_summary_gdf.to_file(catchment_summary_gpkg_path, driver="GPKG")

print(f"Saved patch CSV: {patch_summary_csv_path}")
print(f"Saved patch GeoPackage: {patch_summary_gpkg_path}")
print(f"Saved patch table: {patch_table_path}")
print(f"Saved nearest-neighbour CSV: {nearest_neighbor_summary_csv_path}")
print(f"Saved method summary: {method_summary_csv_path}")
print(f"Saved land-use summary: {land_use_summary_csv_path}")
print(f"Saved parish summary: {parish_summary_csv_path}")
print(f"Saved catchment summary CSV: {catchment_summary_csv_path}")
print(f"Saved catchment summary GeoPackage: {catchment_summary_gpkg_path}")

## Figures

In [ ]:
def style_jamaica_map(axis, title_text):
    axis.set_title(title_text)
    axis.set_axis_off()
    axis.set_aspect("equal")
    Robyn_paper_2_defs.draw_scale_bar(
        axis,
        location=(0.88, 0.78),
        length_km=20,
        linewidth=0.6,
        label_offset=0.02,
        km_offset=0.01,
    )
    Robyn_paper_2_defs.draw_north_arrow(axis, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)


administrative_boundary_handle = Line2D(
    [0],
    [0],
    color="#424242",
    linewidth=0.7,
    label="Administrative boundaries",
)
continuous_legend_settings = {
    "orientation": "horizontal",
    "shrink": 0.62,
    "pad": 0.03,
    "label": "Total carbon potential (tonnes C)",
}

figure, axis = plt.subplots(figsize=(10, 6))
carbon_benefit_summary.plot(
    ax=axis,
    column="total_carbon_potential_with_nn_fill",
    cmap="YlGn",
    legend=True,
    legend_kwds=continuous_legend_settings,
    linewidth=0.08,
    edgecolor="black",
    missing_kwds={"color": "lightgrey", "label": "No estimate"},
)
parishes.boundary.plot(ax=axis, color="#424242", linewidth=0.25)
style_jamaica_map(axis, "Global carbon-benefit raster across existing forest patches")
figure.tight_layout()
figure.savefig(patch_map_path, dpi=300, bbox_inches="tight")
plt.show()

no_coverage_patches = carbon_benefit_summary[~carbon_benefit_summary["has_observed_carbon_benefit"]].copy()
figure, axis = plt.subplots(figsize=(10, 6))
if len(no_coverage_patches) > 0:
    carbon_benefit_summary.plot(ax=axis, color="#E0E0E0", edgecolor="none")
    no_coverage_patches.plot(ax=axis, color="#C62828", edgecolor="black", linewidth=0.2)
    no_coverage_points = no_coverage_patches.set_geometry(no_coverage_patches.representative_point())
    no_coverage_points.plot(ax=axis, color="#C62828", edgecolor="white", linewidth=0.25, markersize=16, marker="o")
else:
    carbon_benefit_summary.plot(ax=axis, color="#BDBDBD", edgecolor="black", linewidth=0.08)
    axis.text(0.5, 0.5, "All patches have valid raster cells", transform=axis.transAxes, ha="center", va="center")
parishes.boundary.plot(ax=axis, color="#424242", linewidth=0.25)
style_jamaica_map(axis, "Existing forest patches with no valid global carbon-benefit cells")
axis.legend(
    handles=[
        Patch(facecolor="#C62828", edgecolor="black", label="No valid raster cells"),
        administrative_boundary_handle,
    ],
    loc="lower left",
    frameon=True,
)
figure.tight_layout()
figure.savefig(no_coverage_map_path, dpi=300, bbox_inches="tight")
plt.show()

figure, axis = plt.subplots(figsize=(10, 6))
if len(nearest_neighbor_fill_gdf) > 0:
    nearest_source_ids = nearest_neighbor_fill_gdf["nearest_source_existing_forest_patch_id"].dropna().astype(int).unique()
    nearest_source_patches = carbon_benefit_summary[
        carbon_benefit_summary["Existing_forest_patch_id"].isin(nearest_source_ids)
    ].copy()
    nearest_source_points = nearest_source_patches.set_geometry(nearest_source_patches.representative_point())
    nearest_fill_points = nearest_neighbor_fill_gdf.set_geometry(nearest_neighbor_fill_gdf.representative_point())
    carbon_benefit_summary.plot(ax=axis, color="#E0E0E0", edgecolor="none")
    nearest_neighbor_links.plot(ax=axis, color="#757575", linewidth=0.3, alpha=0.7)
    nearest_source_patches.plot(ax=axis, color="#1565C0", edgecolor="black", linewidth=0.1, alpha=0.75)
    nearest_neighbor_fill_gdf.plot(ax=axis, color="#C62828", edgecolor="black", linewidth=0.15, alpha=0.9)
    nearest_source_points.plot(ax=axis, color="#1565C0", edgecolor="white", linewidth=0.25, markersize=14, marker="o")
    nearest_fill_points.plot(ax=axis, color="#C62828", edgecolor="white", linewidth=0.25, markersize=14, marker="o")
else:
    carbon_benefit_summary.plot(ax=axis, color="#BDBDBD", edgecolor="black", linewidth=0.08)
    axis.text(0.5, 0.5, "No nearest-neighbour fill required", transform=axis.transAxes, ha="center", va="center")
parishes.boundary.plot(ax=axis, color="#424242", linewidth=0.25)
style_jamaica_map(axis, "Nearest observed carbon-benefit source patches for no-coverage forests")
source_link_legend_handles = [
    Patch(facecolor="#C62828", edgecolor="black", label="No valid cells / filled patch"),
    Patch(facecolor="#1565C0", edgecolor="black", label="Nearest observed source patch"),
    Line2D([0], [0], color="#757575", linewidth=0.8, label="Nearest-neighbour link"),
    administrative_boundary_handle,
]
axis.legend(handles=source_link_legend_handles, loc="lower left", frameon=True)
figure.tight_layout()
figure.savefig(nearest_neighbor_source_map_path, dpi=300, bbox_inches="tight")
plt.show()

top_land_use_summary = land_use_summary.head(10).sort_values("total_carbon_potential_with_nn_fill")
figure, axis = plt.subplots(figsize=(9, 6))
axis.barh(
    top_land_use_summary["Classify"],
    top_land_use_summary["total_carbon_potential_with_nn_fill"] / 1_000_000,
    color="#4CAF50",
    edgecolor="black",
    linewidth=0.4,
)
axis.set_xlabel("Estimated carbon potential (million tonnes C)")
axis.set_ylabel("Existing forest land-use class")
axis.set_title("Existing forest classes by carbon potential")
axis.grid(axis="x", linestyle=":", linewidth=0.4, alpha=0.6)
figure.tight_layout()
figure.savefig(land_use_bar_chart_path, dpi=300, bbox_inches="tight")
plt.show()

figure, axis = plt.subplots(figsize=(10, 6))
catchment_summary_gdf.plot(
    ax=axis,
    column="total_carbon_potential_with_nn_fill",
    cmap="YlGn",
    legend=True,
    legend_kwds=continuous_legend_settings,
    linewidth=0.25,
    edgecolor="black",
    missing_kwds={"color": "lightgrey", "label": "No estimate"},
)
style_jamaica_map(axis, "Existing forest carbon-benefit raster total by paper-2 catchment")
figure.tight_layout()
figure.savefig(catchment_map_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved map: {patch_map_path}")
print(f"Saved no-coverage map: {no_coverage_map_path}")
print(f"Saved nearest-neighbour source map: {nearest_neighbor_source_map_path}")
print(f"Saved land-use bar chart: {land_use_bar_chart_path}")
print(f"Saved catchment map: {catchment_map_path}")